# Random forest and SVM

In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('../data/street_conditions_2025_labeled.csv')
df['created_date'] = pd.to_datetime(df['created_date'])
df['hour'] = df['created_date'].dt.hour
df['day_of_week'] = df['created_date'].dt.day_name()
df['month'] = df['created_date'].dt.month

label = 'closed_48h'
y = df[label]
X = df[['descriptor','open_data_channel_type','hour','day_of_week','month']]
dates = df['created_date']
print(X.shape)
print(y.value_counts())

(70322, 5)
closed_48h
1    44355
0    25967
Name: count, dtype: int64


Same setup as the logistic regression notebook (`models.ipynb`): the same five predictors (`descriptor`, `open_data_channel_type`, `hour`, `day_of_week`, `month`) and the same target `closed_48h`. We keep `created_date` as `dates` so we can split by time instead of randomly.

Splitting and tuning: the function in the next cell follows the logistic regression template so all models are compared the same way.

- November–December is the test set and stays untouched. Development data ends 48 hours before November 1, so no request in development can still be open when the test period starts.
- The validation folds are August, September, and October. Each fold trains only on rows created more than 48 hours before its validation month starts. The fold sizes printed below match the logistic regression notebook (45,215 / 51,024 / 56,157 training rows).
- The one-hot encoder is inside the pipeline, so `GridSearchCV` fits a fresh encoder on each fold's training rows. This does the same job as `clone(preprocessor)` in the logistic regression notebook.
- Scoring is macro F1, so both classes count equally.

In [2]:
def ML_pipeline_time(X,y,dates,ML_algo,param_grid):
    test_start = pd.Timestamp('2025-11-01')
    other_mask = dates < test_start - pd.Timedelta(hours=48)
    test_mask = dates >= test_start
    X_other, y_other = X[other_mask].reset_index(drop=True), y[other_mask].reset_index(drop=True)
    dates_other = dates[other_mask].reset_index(drop=True)
    X_test, y_test = X[test_mask], y[test_mask]

    folds = []
    for month_start in ['2025-08-01','2025-09-01','2025-10-01']:
        start = pd.Timestamp(month_start)
        end = start + pd.DateOffset(months=1)
        i_train = np.where(dates_other < start - pd.Timedelta(hours=48))[0]
        i_val = np.where((dates_other >= start) & (dates_other < end))[0]
        folds.append((i_train,i_val))
        print(start.strftime('%B'),'train:',len(i_train),'val:',len(i_val))

    cat_ftrs = ['descriptor','open_data_channel_type','hour','day_of_week','month']
    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(sparse_output=False,handle_unknown='ignore'))])
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer, cat_ftrs)])
    pipe = make_pipeline(preprocessor,ML_algo)
    grid = GridSearchCV(pipe, param_grid=param_grid,scoring='f1_macro',cv=folds,
                        return_train_score=True,n_jobs=-1,verbose=True)
    grid.fit(X_other, y_other)
    return grid, X_test, y_test

## Random forest

Random forest: we tune `max_depth` (how deep each tree can grow) and `min_samples_split` (how many rows a node needs before it can split again). That is 3 × 6 = 18 combinations × 3 folds = 54 fits. `n_estimators=100` and `random_state=42` stay fixed so the results can be reproduced.

In [3]:
param_grid = {'randomforestclassifier__max_depth': [10,30,100],
              'randomforestclassifier__min_samples_split': [2, 4, 8, 16, 32, 64]}
rf_grid, X_test, y_test = ML_pipeline_time(X,y,dates,RandomForestClassifier(n_estimators=100,random_state=42),param_grid)
print('best model parameters:',rf_grid.best_params_)
print('validation score:',rf_grid.best_score_)

August train: 45215 val: 5590
September train: 51024 val: 5411
October train: 56157 val: 4546
Fitting 3 folds for each of 18 candidates, totalling 54 fits


best model parameters: {'randomforestclassifier__max_depth': 100, 'randomforestclassifier__min_samples_split': 16}
validation score: 0.6838660240877156


In [4]:
rf_results = pd.DataFrame(rf_grid.cv_results_)
rf_results = rf_results[['params','split0_test_score','split1_test_score','split2_test_score',
                         'mean_test_score','std_test_score','mean_train_score']]
rf_results.columns = ['params','August','September','October','mean val f1','std val f1','mean train f1']
rf_results = rf_results.sort_values('mean val f1',ascending=False)
rf_results.to_csv('../results/rf_cv_results.csv',index=False)
rf_results.round(4)

,params,August,September,October,mean val f1,std val f1,mean train f1
15,"{'randomforestclassifier__max_depth': 100, 'ra...",0.6773,0.6861,0.6881,0.6839,0.0047,0.7666
6,"{'randomforestclassifier__max_depth': 30, 'ran...",0.6797,0.6842,0.6866,0.6835,0.0029,0.7941
8,"{'randomforestclassifier__max_depth': 30, 'ran...",0.6812,0.6834,0.6855,0.6834,0.0017,0.7766
12,"{'randomforestclassifier__max_depth': 100, 'ra...",0.6797,0.6836,0.6857,0.6830,0.0025,0.7964
13,"{'randomforestclassifier__max_depth': 100, 'ra...",0.6797,0.6822,0.6871,0.6830,0.0031,0.7944
14,"{'randomforestclassifier__max_depth': 100, 'ra...",0.6785,0.6809,0.6871,0.6822,0.0036,0.7829
7,"{'randomforestclassifier__max_depth': 30, 'ran...",0.6800,0.6812,0.6852,0.6821,0.0022,0.7888
10,"{'randomforestclassifier__max_depth': 30, 'ran...",0.6790,0.6814,0.6856,0.6820,0.0027,0.7493
17,"{'randomforestclassifier__max_depth': 100, 'ra...",0.6779,0.6780,0.6874,0.6811,0.0044,0.7369
16,"{'randomforestclassifier__max_depth': 100, 'ra...",0.6785,0.6780,0.6868,0.6811,0.0040,0.7518


`max_depth = 100` with `min_samples_split = 16` gave the best mean macro F1, **0.6839**, with a standard deviation of **0.0047**.

- Depth 10 was clearly worse (0.652–0.655), so shallow trees underfit.
- At depths 30 and 100, every setting scored within about 0.004 of the best, so the exact pick does not matter much.
- Train F1 (0.74–0.80) is well above validation F1 (about 0.68), so the deeper forests fit the training months more closely than they generalize.

The full grid is saved to `results/rf_cv_results.csv`.

## SVM

SVM with the default RBF kernel. `C` sets how hard the model is penalized for misclassifying training rows (higher C = less regularization). `gamma` sets how far each training point's influence reaches (higher gamma = a more local, more flexible boundary). That is 3 × 3 = 9 combinations × 3 folds = 27 fits.

In [5]:
param_grid = {'svc__C': [0.1, 1, 10],
              'svc__gamma': [0.01, 0.1, 1]}
svc_grid, X_test, y_test = ML_pipeline_time(X,y,dates,SVC(),param_grid)
print('best model parameters:',svc_grid.best_params_)
print('validation score:',svc_grid.best_score_)

August train: 45215 val: 5590
September train: 51024 val: 5411
October train: 56157 val: 4546
Fitting 3 folds for each of 9 candidates, totalling 27 fits


best model parameters: {'svc__C': 1, 'svc__gamma': 1}
validation score: 0.6843628506320973


In [6]:
svc_results = pd.DataFrame(svc_grid.cv_results_)
svc_results = svc_results[['params','split0_test_score','split1_test_score','split2_test_score',
                           'mean_test_score','std_test_score','mean_train_score']]
svc_results.columns = ['params','August','September','October','mean val f1','std val f1','mean train f1']
svc_results = svc_results.sort_values('mean val f1',ascending=False)
svc_results.to_csv('../results/svc_cv_results.csv',index=False)
svc_results.round(4)

,params,August,September,October,mean val f1,std val f1,mean train f1
5,"{'svc__C': 1, 'svc__gamma': 1}",0.6795,0.6872,0.6863,0.6844,0.0034,0.7901
8,"{'svc__C': 10, 'svc__gamma': 1}",0.6753,0.6866,0.6845,0.6821,0.0049,0.7952
7,"{'svc__C': 10, 'svc__gamma': 0.1}",0.6802,0.6818,0.6811,0.6811,0.0006,0.7341
4,"{'svc__C': 1, 'svc__gamma': 0.1}",0.6759,0.6709,0.6714,0.6727,0.0022,0.7034
6,"{'svc__C': 10, 'svc__gamma': 0.01}",0.6511,0.6492,0.6389,0.6464,0.0054,0.6717
3,"{'svc__C': 1, 'svc__gamma': 0.01}",0.6501,0.6435,0.6437,0.6458,0.0030,0.6724
1,"{'svc__C': 0.1, 'svc__gamma': 0.1}",0.6487,0.6453,0.6368,0.6436,0.0050,0.6649
2,"{'svc__C': 0.1, 'svc__gamma': 1}",0.6351,0.6355,0.6372,0.6359,0.0009,0.6887
0,"{'svc__C': 0.1, 'svc__gamma': 0.01}",0.6369,0.6204,0.6179,0.6251,0.0084,0.6512


`C = 1` with `gamma = 1` gave the best mean macro F1, **0.6844**, with a standard deviation of **0.0034**.

- At C = 1, the score rises with every increase in gamma, and gamma = 1 is the largest value we tried. A larger gamma (for example 3 or 10) might do slightly better, so it is worth one more run if time allows.
- C = 0.1 gave the three lowest scores, so the model needs to be allowed to fit more closely than that.

The full grid is saved to `results/svc_cv_results.csv`.

In [7]:
summary = pd.DataFrame({'model':['random forest','SVM'],
                        'best params':[rf_grid.best_params_,svc_grid.best_params_],
                        'mean val f1':[rf_grid.best_score_,svc_grid.best_score_],
                        'std val f1':[rf_grid.cv_results_['std_test_score'][rf_grid.best_index_],
                                      svc_grid.cv_results_['std_test_score'][svc_grid.best_index_]]})
summary.round(4)

,model,best params,mean val f1,std val f1
0,random forest,"{'randomforestclassifier__max_depth': 100, 'ra...",0.6839,0.0047
1,SVM,"{'svc__C': 1, 'svc__gamma': 1}",0.6844,0.0034


READ MARKDOWN FOR TEAM:

| Model | Best parameters | Mean validation macro F1 | Std |
|---|---|---|---|
| Logistic regression (`models.ipynb`) | C = 0.1 | 0.6649 | 0.0031* |
| Random forest | max_depth = 100, min_samples_split = 16 | **0.6839** | 0.0047 |
| SVM (RBF) | C = 1, gamma = 1 | **0.6844** | 0.0034 |

\*Population standard deviation, computed from the three fold scores in `models.ipynb` so that it matches how `GridSearchCV` reports `std_test_score`. The 0.0038 in `models.ipynb` is the sample standard deviation of the same scores.

- Random forest and SVM both beat logistic regression by about 0.019 macro F1.
- SVM is ahead of random forest by only 0.0005, much less than either model's fold-to-fold spread, so on validation the two are effectively tied.
- Same template as logistic regression: August–October chronological folds, the same 48-hour gaps, and macro F1.

November–December is still untouched. We have not refit either model on the full development set or scored the test set; that waits until every model is done.